# Phase 1 chunk 6: Training Loop Mastery.


## Part A: The Clinical Environment - Engineering the Perfect Engine

* **Objective**: To engineer a flawless, reusable, professional training and validation script. We are not focused on state-of-the-art accuracy; we are focused on **state-of-the-art code quality.**

* **Dataset**: FashionMNIST. Why? It's the perfect mannequin. It's clean, balanced, loads instantly, and requires a simple model. It allows us to focus entirely on the engineering of the loop without being distracted by data problems.

**Theory-to-Practice Bridge: Iterative Optimization & Generalization**
* **Theory (Iterative Optimization)**: 

    You know that Gradient Descent is an iterative process. You repeatedly calculate a gradient and take a step. The training loop is the for loop that implements this iteration over and over, showing the model batches of data until it has seen the entire dataset. One full pass over the dataset is called an epoch.

* **Theory (Generalization)**: 

    A model is useless if it only memorizes the training data. We need it to generalize to new, unseen data. The **validation loop** is our check for generalization. After each epoch, we pause training, put the model in evaluation mode (`model.eval()`), and measure its performance on a separate validation set without updating any weights. If training loss goes down but validation loss goes up, we are **overfitting.**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time


# 1.  Configuration
# for reproducibility
torch.manual_seed(42)

# Hyperparameters
LEARNING_RATE  = 0.001
BATCH_SIZE = 64
EPOCHS = 5

# Set device
device = "cuda" if torch.cuda.is_available() else 'cpu'
print(f"Using the device : {device}")

# 2. Loading the FashionMNIST dataset
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5),(0.5))
    ]
)

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True,
                                      transform=transform)

test_dataset =  datasets.FashionMNIST(root='./data',
                                      train=False,
                                      download=True,
                                      transform=transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
    )

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Define the model
class FashionMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.net = nn.Sequential(
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.net(self.flatten(x))

model = FashionMNIST().to(device)

# 4. Loss and Optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = LEARNING_RATE)

# 5. The Training and Validation Functions

def train_one_epoch(loader, model, loss_fn, optimizer):
    model.train() # Set model for training mode
    running_loss = 0.0
    for batch, (X, y) in enumerate(loader):
        X, y = X.to(device), y.to(device)
        
        # 1. Forward pass
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    
    return running_loss / len(loader)


def validate_one_epoch(loader, model, loss_fn):
    model.eval() # set model to evaluation mode
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            pred = model(X)

            running_loss += loss_fn(pred, y).item()


            # calculate accuracy
            total += y.size(0)
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    avg_loss = running_loss / len(loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


# --- 6. The Main Training Loop ---
print("Starting training...")
start_time = time.time()

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(train_loader, model, loss_fn, optimizer)
    val_loss, val_acc = validate_one_epoch(test_loader, model, loss_fn)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

end_time = time.time()
print(f"Finished training in {end_time - start_time:.2f} seconds.")



Using the device : cuda
Starting training...
Epoch 1/5 | Train Loss: 0.5123 | Val Loss: 0.4196 | Val Acc: 84.53%
Epoch 2/5 | Train Loss: 0.3784 | Val Loss: 0.3967 | Val Acc: 85.44%
Epoch 3/5 | Train Loss: 0.3406 | Val Loss: 0.4100 | Val Acc: 85.44%
Epoch 4/5 | Train Loss: 0.3177 | Val Loss: 0.3583 | Val Acc: 86.93%
Epoch 5/5 | Train Loss: 0.2983 | Val Loss: 0.3326 | Val Acc: 88.29%
Finished training in 169.90 seconds.


# Explorations (Clinical Environment)

* **Beginner**: 

    Change the EPOCHS constant from 5 to 15. Observe the training and validation loss. Do you see signs of overfitting (validation loss starting to increase while 
    training loss continues to decrease)?

* **Intermediate**: 

    Add code to save your model's weights after training is complete. Use torch.save(model.state_dict(), 'fashion_mnist_model.pth'). Why is saving the state_dict generally better than saving the entire model object?
    
* **Advanced**: 

    Implement early stopping. Modify the main loop to track the best validation accuracy seen so far. Add a "patience" counter. If the validation accuracy doesn't improve for, say, 3 epochs in a row, break the training loop. This is a crucial technique to prevent overfitting and save compute time.
